# 04 — Field-aware tokenization
Challenge: financial events have heterogeneous numeric, categorical and textual fields. Upstream solves this with key/value IDs in a shared vocabulary, within-field positions, per-key percentile bins, BPE and separate temporal features. This is learning, not a new contribution. Run from repo root with experiment 03 artifacts (or use runner).

In [ ]:
from pathlib import Path
from pragmatiq.data.tokenizer import PragmaTokenizer, iter_user_records, time_encode
root=Path('runs/research_decomposed')
tok=PragmaTokenizer.load(root/'tok/tokenizer')
print(tok.vocab_size, tok.config)
print(tok.field_kind)

## Numeric bins
A dedicated zero bucket plus quantile intervals fitted per numeric key; values within a bin share one value ID. This loses within-bin magnitude precision. Low-cardinality numeric codes may be categorical; use explicit schema overrides when necessary.

In [ ]:
import numpy as np
for k,b in list(tok.binners.items())[:2]:
    x=np.array([-100.,0.,1.,10.,100.,1000.])
    ids=b.transform(x)
    print(k, list(zip(x.tolist(),ids.tolist(),[b.bucket_repr(int(i)) for i in ids])))

## Trace one transaction
Each field contributes parallel key/value/position arrays. CSR event offsets recover each event's slice; time_log is relative to the latest event, not inter-event delta. Calendar features are separate.

In [ ]:
raw=next(iter_user_records(root/'raw'))
r=tok.encode(raw)
i=next(i for i,e in enumerate(raw.events) if e[1]=='transaction')
lo,hi=map(int,r.event_offsets[i:i+2])
print(raw.events[i])
print(r.key_ids[lo:hi],r.value_ids[lo:hi],r.positions[lo:hi])
print(tok.decode_event(r,i))
print(r.event_offsets.shape,r.event_ts.shape,r.key_ids.shape)
print(r.time_log[i],r.hour[i],r.dow[i],r.dom[i])

## Measure token inflation
Compare events vs actual event tokens; batching by events alone does not guarantee equal token workloads. BPE can expand text into multiple tokens. Use aggregate metrics only.

In [ ]:
%run research/experiments/04_tokenizer_analysis.py

## Research later
Investigate continuous numeric embeddings and high-cardinality merchant IDs as extensions after the learning phase. Fit vocabularies on training-only chronological data for any predictive benchmark. Do not infer monosemantic concepts from token IDs alone.